In [1]:
# ---- MOVED CODE FROM PREVIOUS DOC AS DISCUSSED. PLEASE PLACE IN DOC WHEREVER YOU PREFER.
# ---- I renamed the df from "df_encoded" to "df" since it should use the cleaned csv file.

# Encode regionidzip using KMeans:
# compute mean of related column per category:
cat_means = df.groupby("regionidzip")["taxvaluedollarcnt"].mean()
# map means back and cluster into 10 groups:
df["regionidzip"] = df["regionidzip"].map(cat_means)
kmeans = KMeans(n_clusters=10, random_state=RANDOM_STATE)
df["regionidzip"] = kmeans.fit_predict(df[["regionidzip"]])

# Encode propertycountylandusecode using KMeans:
# compute mean of related column per category:
cat_means = df.groupby("propertycountylandusecode")["taxvaluedollarcnt"].mean()
#  map means back and cluster into 10 groups:
df["propertycountylandusecode"] = df["propertycountylandusecode"].map(cat_means)
kmeans = KMeans(n_clusters=10, random_state=RANDOM_STATE)
df["propertycountylandusecode"] = kmeans.fit_predict(df[["propertycountylandusecode"]])

NameError: name 'df' is not defined

# 06 - Feature Engineering

**Milestone 1 — Part 5**: Investigate transformations to better expose data patterns.

## Objectives
- Apply feature scaling (standardize / normalize)
- Try at least **3 transformations** (log, polynomial, ratios, etc.)
- Evaluate each using correlations, F-scores, or feature selection
- Write transformations as **reusable functions**

## Expected Outcomes
| Deliverable | Description |
|---|---|
| Scaling function | StandardScaler applied to numerical features |
| Transform 1 | e.g., log(sqft) — evaluated with correlation |
| Transform 2 | e.g., price_per_sqft ratio — evaluated with F-score |
| Transform 3 | e.g., polynomial bedroom*bathroom — evaluated |
| Comparison table | Before vs after correlation/F-score for each |

## Key Rule (from assignment)
> Write transformations as functions. Don't commit yet — models may respond differently.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_regression
from sklearn.cluster import KMeans

RANDOM_STATE = 42
TARGET = "taxvaluedollarcnt"

df = pd.read_csv("zillow_cleaned.csv")
print(f"Shape: {df.shape}")

## Helper: Evaluate a new feature's relationship with the target

In [ ]:
def evaluate_feature(dataframe: pd.DataFrame, feature_name: str, target: str = TARGET):
    """Print correlation and F-score for a single feature vs target."""
    corr = dataframe[[feature_name, target]].corr().iloc[0, 1]
    f_score, p_val = f_regression(dataframe[[feature_name]], dataframe[target])
    print(f"  {feature_name:40s}  corr={corr:+.4f}   F={f_score[0]:12.1f}   p={p_val[0]:.2e}")
    return {"feature": feature_name, "corr": corr, "f_score": f_score[0], "p_value": p_val[0]}

## 1. Feature Scaling

In [ ]:
def scale_features(dataframe: pd.DataFrame, exclude_cols: list = None) -> pd.DataFrame:
    """Standardize numerical features (zero mean, unit variance)."""
    if exclude_cols is None:
        exclude_cols = []
    num_cols = [c for c in dataframe.select_dtypes(include=[np.number]).columns if c not in exclude_cols]
    scaler = StandardScaler()
    result = dataframe.copy()
    result[num_cols] = scaler.fit_transform(result[num_cols])
    return result, scaler

df_scaled, scaler = scale_features(df, exclude_cols=[TARGET])
print("Scaling complete. Sample means (should be ~0):")
print(df_scaled.drop(columns=[TARGET]).mean().head(5))

## 2. Transformation 1 — Log Transform

Applying `log1p` to right-skewed area/size features to reduce skew and potentially improve linear correlation.

In [ ]:
def add_log_features(dataframe: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Add log1p-transformed versions of specified columns."""
    result = dataframe.copy()
    for col in columns:
        if col in result.columns:
            result[f"log_{col}"] = np.log1p(result[col].clip(lower=0))
    return result

# TODO: Adjust column list based on skewed features from notebook 02
log_cols = ["calculatedfinishedsquarefeet", "lotsizesquarefeet", "taxvaluedollarcnt"]
df_eng = add_log_features(df, log_cols)

print("Before vs After log transform:")
for col in log_cols:
    if col != TARGET:
        evaluate_feature(df, col)
        evaluate_feature(df_eng, f"log_{col}")

## 3. Transformation 2 — Ratio Features

Ratios can capture per-unit relationships that raw features miss.

In [ ]:
def add_ratio_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Add ratio-based engineered features."""
    result = dataframe.copy()
    if "calculatedfinishedsquarefeet" in result.columns and "lotsizesquarefeet" in result.columns:
        result["living_to_lot_ratio"] = (
            result["calculatedfinishedsquarefeet"] / result["lotsizesquarefeet"].replace(0, np.nan)
        ).fillna(0)

    if "bathroomcnt" in result.columns and "bedroomcnt" in result.columns:
        result["bath_per_bedroom"] = (
            result["bathroomcnt"] / result["bedroomcnt"].replace(0, np.nan)
        ).fillna(0)

    # TODO: Add more ratios that make business sense
    return result

df_eng = add_ratio_features(df_eng)

print("Ratio feature evaluation:")
for feat in ["living_to_lot_ratio", "bath_per_bedroom"]:
    if feat in df_eng.columns:
        evaluate_feature(df_eng, feat)

## 4. Transformation 3 — Polynomial / Interaction Features

Multiplying or squaring features to capture nonlinear relationships.

In [ ]:
def add_polynomial_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Add polynomial and interaction features."""
    result = dataframe.copy()

    if "calculatedfinishedsquarefeet" in result.columns:
        result["sqft_squared"] = result["calculatedfinishedsquarefeet"] ** 2

    if "bathroomcnt" in result.columns and "bedroomcnt" in result.columns:
        result["bath_x_bed"] = result["bathroomcnt"] * result["bedroomcnt"]

    # TODO: Add more interactions based on domain knowledge
    return result

df_eng = add_polynomial_features(df_eng)

print("Polynomial feature evaluation:")
for feat in ["sqft_squared", "bath_x_bed"]:
    if feat in df_eng.columns:
        evaluate_feature(df_eng, feat)

## 5. Comparison Summary

In [ ]:
# Collect all new features for a side-by-side comparison
new_features = [
    col for col in df_eng.columns
    if col not in df.columns and col != TARGET
]

results = []
print(f"{'Feature':40s}  {'Corr':>8s}  {'F-Score':>12s}  {'p-value':>10s}")
print("-" * 75)
for feat in new_features:
    results.append(evaluate_feature(df_eng, feat))

results_df = pd.DataFrame(results).sort_values("f_score", ascending=False)
results_df

---
## Discussion 5

Describe why you chose these transformations and what you observed:
- Which transformations improved correlation / F-score?
- Which didn't help?
- What would you recommend carrying forward to modeling?

*YOUR ANSWER HERE*

---
### Next Notebook → `07_executive_summary.ipynb`